This is an example notebook to run ACOLITE processing on the Copernicus Data Space Ecosystem JupyterLab service. This notebook is currently configured to run ACOLITE/DSF processing for a Sentinel-3 OLCI scene, but can be configured to do most of the processing options in ACOLITE. For more than occasional processing it is recommended to set up a local environment and install the software on your system. Please note that all data (including LUTs) and installed software in the JupyterLab instance are deleted after it expires. 

In this example, Sentinel-3 scenes are retrieved from the Copernicus Data Space Ecosystem (CDSE) and hence requires CDSE credentials to be configured. Download can take up to several minutes. The first run for a specific sensor will be slower as well, as the sensor specific lookup tables (LUTs) are retrieved from GitHub. Optional ancillary data can be used if EarthData credentials are also set up. Register at the EarthData Login page and select OB.DAAC Data Access and LP DAAC Data Pool in your profile under Authorised Apps, Approve More Applications.

To run the code in the notebook, click the execute button next to each cell to run that particular cell. Since cells use outputs from preceding cells, please run them in order.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be Created: 2026-05-21

The first cell verifies whether the notebook is running in the "Geo Science" environment, which should have been selected automatically. If the environment is incorrect, please change it using the "Select Kernel" menu. This can be accessed from the "Kernel" dropdown menu and selecting "Change Kernel..." or directly from the top right of the notebook, which should be labeled with the current environment name, e.g. "Python 3 (ipykernel)". The acolite code is cloned from the GitHub repository if it is not already present.

In [ ]:
import os, sys
if sys.prefix != '/opt/conda/envs/geo':
    raise BaseException('Please change conda kernel to "Geo Science"')
    
## clone acolite from github
if not os.path.exists('acolite'):
    !git clone --depth 1 https://github.com/acolite/acolite

## add acolite clone to path and import
sys.path.append('acolite')
import acolite as ac

The second cell loads your credentials from a plain text credentials.txt file. Please enter at least CDSE_u (your CDSE username) and CDSE_p (your CDSE password) for accessing data files from the CDSE.

In [ ]:
## load credentials and set as env variables
if not os.path.exists('credentials.txt'):
    raise BaseException('Please upload your credentials.txt file to the current environment')
credentials = ac.shared.import_config('credentials.txt')
for k in credentials: os.environ[k] = credentials[k]

The next cell sets up the region of interest for querying the CDSE for available scenes. Here we use a WKT polygon which contains the boundaries of our region of interest.

In [ ]:
## location and date of interest
## you can use the Copernicus browser to find images
## https://browser.dataspace.copernicus.eu/
region_name = 'PoRiverPlume'
wkt = "POLYGON ((13.00 45.25, 12.00 45.25, 12.00 44.50, 13.00 44.50, 13.00 45.25))"
date = '2025-10-28'

The next cell queries and downloads scenes from CDSE, to create a local copy in the Input directory. Please note that the download can take several minutes per scene.

In [ ]:
## use ACOLITE API to query CDSE
urls, scenes = ac.api.cdse.query(roi = wkt, collection = "SENTINEL-3", timeliness = 'NT', 
                                 start_date = date, end_date = date)
## print the results
for scene in scenes: print(scene)

## use ACOLITE API to download the scenes from CDSE
local_scenes = ac.api.cdse.download(urls, scenes = scenes, output = 'Input/')

The following cell sets up a Python dictionary with ACOLITE processing settings, that can be passed to the main processing function. Here we use DSF processing for the region of interest defined by the WKT string above. As we will reproject to an UTM projection, an automatic 0.1 degree buffer is added to the region of interest to improve coverage in the corners of the output file.

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': local_scenes,
            'output': 'Output/',

            ## set up region of interest based on wkt or other polygon
            'region_name': region_name,
            'polygon': wkt,

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## add a buffer for reprojection
            'limit_buffer': 0.1,
            'limit_buffer_units': 'degree',

            ## reproject to UTM
            'output_projection': True,
            'reproject_before_ac': True,

            ## enable compression of NetCDF files
            'netcdf_compression': True,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

Since we will reproject the OLCI data, we need to have pyresample installed. The next cell verifies whether pyresample is available and installs it using pip if it is not. This is a cell specifically for the "Geo Science" environment on CDSE JupyterLab, and this is typically not necessary as the package will be installed in the acolite environment.

In [ ]:
try:
    import pyresample
except:
    !pip install pyresample

The final cell runs the actual processing with the specified settings.

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)